# 05 · BioMistral-7B QLoRA Fine-Tuning on Medical Conversations

Fine-tunes **BioMistral-7B** with QLoRA (4-bit NF4) on the integrated medical conversation dataset.

**Kaggle setup:**
- Attach `integrated_conversations.csv` as a dataset and update `DATASET_SLUG` below.
- Enable **GPU T4 x2** or **P100** in notebook settings.
- Enable **Internet** to install packages.
- Output adapter is saved to `/kaggle/working/biomistral_qlora_20k/`.

## 0 · Environment detection & paths

In [ ]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    DATASET_SLUG = 'shilpadevi1527/integrated-conversations-csv'
    INPUT_CSV  = Path(f'/kaggle/input/datasets/{DATASET_SLUG}/integrated_conversations.csv')
    OUTPUT_DIR = Path('/kaggle/working/biomistral_qlora_20k')
else:
    DATA_DIR   = Path('../data/processed')
    INPUT_CSV  = DATA_DIR / 'integrated_conversations.csv'
    OUTPUT_DIR = Path('./biomistral_qlora_20k')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Environment : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Input CSV   : {INPUT_CSV}')
print(f'Output dir  : {OUTPUT_DIR}')

if IS_KAGGLE:
    for dirpath, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirpath, filename))

## 1 · Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes

## 2 · GPU check

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')
else:
    print('No GPU detected — a GPU accelerator is required for this notebook.')

## 3 · Library version check

In [ ]:
import transformers
import datasets
import peft
import accelerate
import bitsandbytes

print('Transformers:', transformers.__version__)
print('Datasets    :', datasets.__version__)
print('PEFT        :', peft.__version__)
print('Accelerate  :', accelerate.__version__)
print('BitsAndBytes:', bitsandbytes.__version__)

## 4 · Load & inspect dataset

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV, low_memory=False)
print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

In [ ]:
print(df.head(10).to_string())

In [ ]:
print('Speaker distribution:')
print(df['speaker'].value_counts())

print('\nSource distribution:')
print(df['source_dataset'].value_counts())

print('\nDialogue origin:')
print(df['dialogue_origin'].value_counts())

print('\nMissing values:')
print(df.isnull().sum())

## 5 · Missing utterance audit

In [ ]:
missing_rows = df[df['utterance'].isna()]

print('Number of missing utterances:', len(missing_rows))
print('\nMissing utterances by speaker:')
print(missing_rows['speaker'].value_counts())
print('\nMissing utterances by source:')
print(missing_rows['source_dataset'].value_counts())
print('\nMissing utterance rows:')
print(missing_rows[
    ['dialogue_id', 'turn_id', 'speaker', 'source_dataset', 'original_id']
].to_string(index=False))

In [ ]:
affected_dialogues = df.loc[df['utterance'].isna(), 'dialogue_id'].unique()
print('Number of affected conversations:', len(affected_dialogues))

missing_by_dialogue = (
    df[df['dialogue_id'].isin(affected_dialogues)]
    .groupby(['dialogue_id', 'source_dataset'])['utterance']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='missing_turns')
)
print('\nMissing turns per affected conversation:')
print(missing_by_dialogue.to_string(index=False))

In [ ]:
for dialogue_id in affected_dialogues[:5]:
    print('\n' + '=' * 80)
    print('DIALOGUE:', dialogue_id)
    temp = df[df['dialogue_id'] == dialogue_id].sort_values('turn_id')
    print(temp[['turn_id', 'speaker', 'utterance', 'source_dataset']].to_string(index=False))

## 6 · Data cleaning

In [ ]:
df_clean = df.dropna(subset=['utterance']).copy()
print('Original shape:', df.shape)
print('Cleaned shape :', df_clean.shape)
print('Remaining missing utterances:', df_clean['utterance'].isna().sum())

In [ ]:
empty_utterances = df_clean['utterance'].astype(str).str.strip().eq('').sum()
print('Empty/blank utterances:', empty_utterances)

In [ ]:
turns_per_dialogue = df_clean.groupby('dialogue_id').size()
print('Total dialogues:', turns_per_dialogue.shape[0])
print('\nTurn count distribution:')
print(turns_per_dialogue.value_counts().sort_index())
print('\nMinimum turns:', turns_per_dialogue.min())
print('Maximum turns:', turns_per_dialogue.max())

In [ ]:
invalid_dialogues = []
for dialogue_id, group in df_clean.groupby('dialogue_id'):
    speakers = group.sort_values('turn_id')['speaker'].tolist()
    for i in range(1, len(speakers)):
        if speakers[i] == speakers[i - 1]:
            invalid_dialogues.append(dialogue_id)
            break
print('Dialogues with consecutive same speakers:', len(set(invalid_dialogues)))

In [ ]:
df_check = df_clean.sort_values(['dialogue_id', 'turn_id']).copy()
same_as_previous = (
    df_check['speaker'] == df_check.groupby('dialogue_id')['speaker'].shift(1)
)
affected_dialogues = df_check.loc[same_as_previous, 'dialogue_id'].unique()
print('Dialogues with consecutive same speakers:', len(affected_dialogues))

In [ ]:
for dialogue_id in affected_dialogues[:10]:
    temp = df_check[df_check['dialogue_id'] == dialogue_id]
    print('\n' + '=' * 80)
    print('Dialogue ID:', dialogue_id)
    print(temp[['turn_id', 'speaker', 'utterance', 'source_dataset']].to_string(index=False))

In [ ]:
turn_counts = (
    df_clean.groupby('dialogue_id')['turn_id']
    .agg(['min', 'max', 'count'])
)
turn_counts['expected_count'] = turn_counts['max'] - turn_counts['min'] + 1
gapped_dialogues = turn_counts[turn_counts['count'] != turn_counts['expected_count']]
print('Dialogues with turn_id gaps:', len(gapped_dialogues))
print('Total dialogues:', len(turn_counts))

## 7 · Master dataset & text statistics

In [ ]:
df_master = df_clean.copy()
print('Master dataset shape:', df_master.shape)
print('Missing utterances  :', df_master['utterance'].isna().sum())
print('Empty utterances    :', df_master['utterance'].astype(str).str.strip().eq('').sum())

In [ ]:
df_master['text_length'] = df_master['utterance'].astype(str).str.len()
print('Average characters:', round(df_master['text_length'].mean(), 2))
print('Median characters :', df_master['text_length'].median())
print('Maximum characters:', df_master['text_length'].max())
print('\nTurns by speaker:')
print(df_master['speaker'].value_counts())
print('\nTurns by source:')
print(df_master['source_dataset'].value_counts())

In [ ]:
for limit in [1000, 2000, 4000, 8000, 16000]:
    count = (df_master['text_length'] > limit).sum()
    print(f'Utterances longer than {limit} characters: {count}')

In [ ]:
df_master = df_master.drop(columns=['text_length'])
print('Columns:', df_master.columns.tolist())
print('Shape  :', df_master.shape)

In [ ]:
sample_dialogue_id = df_master['dialogue_id'].iloc[0]
sample = (
    df_master[df_master['dialogue_id'] == sample_dialogue_id]
    .sort_values('turn_id')
)
print('Dialogue ID   :', sample_dialogue_id)
print('Number of turns:', len(sample))
for _, row in sample.iterrows():
    print(f"\n{row['speaker'].upper()}:")
    print(row['utterance'])

In [ ]:
dialogue_sources = (
    df_master.groupby(['dialogue_id', 'source_dataset'])
    .size()
    .reset_index(name='turn_count')
)
print(dialogue_sources['source_dataset'].value_counts())
print('\nNumber of dialogues by source:')
print(dialogue_sources.groupby('source_dataset')['dialogue_id'].nunique())

In [ ]:
print('Total non-null source labels:', df_master['source_label_raw'].notna().sum())
print('\nUnique source labels:')
print(df_master['source_label_raw'].dropna().value_counts().head(30))
print('\nNumber of unique labels:', df_master['source_label_raw'].nunique())

## 8 · Build conversation-level training examples

In [ ]:
df_conversations = (
    df_master
    .groupby('dialogue_id', sort=False)[['dialogue_id', 'turn_id', 'speaker', 'utterance']]
    .apply(
        lambda group: '\n'.join(
            f"{row['speaker'].capitalize()}: {str(row['utterance']).strip()}"
            for _, row in group.sort_values('turn_id').iterrows()
        )
    )
    .reset_index(name='text')
)
print('Conversation dataset shape:', df_conversations.shape)
print('Missing conversations     :', df_conversations['text'].isna().sum())
print('\nFirst conversation preview:')
print(df_conversations.iloc[0]['text'][:1000])

## 9 · Train / validation split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_conversations,
    test_size=0.10,
    random_state=42,
    shuffle=True
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print('Training conversations  :', len(train_df))
print('Validation conversations:', len(val_df))
print('Training %  :', round(len(train_df) / len(df_conversations) * 100, 2))
print('Validation %:', round(len(val_df)   / len(df_conversations) * 100, 2))

In [ ]:
train_ids = set(train_df['dialogue_id'])
val_ids   = set(val_df['dialogue_id'])
overlap   = train_ids.intersection(val_ids)
print('Training dialogue IDs   :', len(train_ids))
print('Validation dialogue IDs :', len(val_ids))
print('Overlapping dialogue IDs:', len(overlap))

## 10 · Load tokenizer & token-length analysis

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'BioMistral/BioMistral-7B'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print('Tokenizer loaded:', MODEL_NAME)
print('Vocabulary size :', len(tokenizer))
print('PAD token       :', tokenizer.pad_token, '| id:', tokenizer.pad_token_id)
print('EOS token       :', tokenizer.eos_token, '| id:', tokenizer.eos_token_id)

In [ ]:
import numpy as np

sample_texts = train_df['text'].sample(n=min(1000, len(train_df)), random_state=42).tolist()
token_lengths = [
    len(tokenizer(text, add_special_tokens=True)['input_ids'])
    for text in sample_texts
]

print('Conversations checked:', len(token_lengths))
print('Minimum tokens  :', min(token_lengths))
print('Median tokens   :', int(np.median(token_lengths)))
print('Average tokens  :', int(np.mean(token_lengths)))
print('90th percentile :', int(np.percentile(token_lengths, 90)))
print('95th percentile :', int(np.percentile(token_lengths, 95)))
print('Maximum tokens  :', max(token_lengths))

## 11 · Tokenize datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[['text']], preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df[['text']],   preserve_index=False)

print('Training dataset  :', train_dataset)
print('Validation dataset:', val_dataset)
print('\nExample:')
print(train_dataset[0]['text'][:500])

In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc='Tokenizing training data'
)
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc='Tokenizing validation data'
)

print('Training tokenized  :', train_tokenized)
print('Validation tokenized:', val_tokenized)

In [ ]:
print('Token count (first example)    :', len(train_tokenized[0]['input_ids']))
print('Attention mask length          :', len(train_tokenized[0]['attention_mask']))
print('\nFirst 20 token IDs:')
print(train_tokenized[0]['input_ids'][:20])
print('\nFirst 20 attention mask values:')
print(train_tokenized[0]['attention_mask'][:20])

## 12 · Load BioMistral-7B with 4-bit quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

print('BioMistral-7B loaded successfully!')
print('Model device:', model.device)

## 13 · Attach QLoRA adapter

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
print('Model prepared for QLoRA training!')

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 14 · Data collator

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
print('Data collator created.')

## 15 · 10-step sanity check

In [ ]:
import time
from transformers import TrainingArguments, Trainer

sanity_args = TrainingArguments(
    output_dir='./biomistral_sanity',
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=False,
    logging_steps=1,
    eval_strategy='no',
    save_strategy='no',
    report_to='none',
    optim='paged_adamw_8bit',
    remove_unused_columns=False
)

sanity_trainer = Trainer(
    model=model,
    args=sanity_args,
    train_dataset=train_tokenized.select(range(10)),
    data_collator=data_collator,
)

print('Starting 10-step sanity check...')
sanity_result = sanity_trainer.train()
print('Sanity check complete!')
print('Training loss:', sanity_result.training_loss)
print(
    'GPU memory after sanity check:',
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    'GB'
)

In [ ]:
import gc

del sanity_trainer
del sanity_result
gc.collect()
torch.cuda.empty_cache()

print('Sanity check trainer cleared.')
print(
    'GPU memory currently allocated:',
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    'GB'
)

## 16 · 100-step speed test

In [ ]:
TRAIN_SIZE = 20_000
train_small = train_tokenized.shuffle(seed=42).select(range(TRAIN_SIZE))
print('20K training examples :', len(train_small))
print('Validation examples   :', len(val_tokenized))

In [ ]:
speed_args = TrainingArguments(
    output_dir='./biomistral_speed_test',
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=False,
    logging_steps=25,
    eval_strategy='no',
    save_strategy='no',
    report_to='none',
    optim='paged_adamw_8bit',
    remove_unused_columns=False
)

speed_trainer = Trainer(
    model=model,
    args=speed_args,
    train_dataset=train_small,
    data_collator=data_collator,
)

print('Starting 100-step speed test...')
start_time = time.time()
speed_result = speed_trainer.train()
elapsed = time.time() - start_time

print('100-step test completed!')
print('Time taken          :', round(elapsed / 60, 2), 'minutes')
print('Average time/step   :', round(elapsed / 100, 2), 'seconds')
print('Training loss       :', speed_result.training_loss)

## 17 · Full 20K training run

In [ ]:
from transformers import TrainingArguments

final_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=False,
    logging_steps=100,
    eval_strategy='no',
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    optim='paged_adamw_8bit',
    remove_unused_columns=False
)
print('Final 20K training configuration ready!')

In [ ]:
from transformers import Trainer

final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=train_small,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print('Trainer created successfully!')
print('Training examples  :', len(train_small))
print('Validation examples:', len(val_tokenized))

In [ ]:
print('Starting BioMistral-7B QLoRA training on 20,000 conversations...')
print('Training examples:', len(train_small))
train_result = final_trainer.train()
print('\nTraining completed!')
print('Final training loss:', train_result.training_loss)

## 18 · Inspect model & run inference

In [ ]:
print('Model           :', MODEL_NAME)
print('Device          :', model.device)
print('Total parameters:', sum(p.numel() for p in model.parameters()))
print('LoRA attached   :', hasattr(model, 'peft_config'))

In [ ]:
import torch

question = 'What are the common symptoms of iron deficiency anemia?'
prompt   = f'User: {question}\nAssistant:'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('INPUT:')
print(question)
print('\nMODEL OUTPUT:')
print(response)